In [54]:
import pandas as pd
import gspread

from sqlalchemy import create_engine
from urllib.parse import quote_plus
from pymongo import MongoClient
from oauth2client.service_account import ServiceAccountCredentials

SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1BJSaVrXHnFXzqp1EVRj94-jqth8s86etTXW67H0iYgE/edit?usp=sharing"
GOOGLE_JSON_FILE = "google_service_account.json"

# mysql 연결
db_user = "root"
db_password = quote_plus("akzpxld12#4")
df_host = "localhost"
db_name = "wconcept_db_260423"

engine = create_engine(
    f"mysql+pymysql://{db_user}:{db_password}@{df_host}/{db_name}?charset=utf8mb4"
)

# mongodb 연결
mongo_client = MongoClient("mongodb://localhost:27017/")
mongo_db = mongo_client["wconcept_db_260423"]
blog_collection = mongo_db["blog_posts"]

# brand_summary - 1
brand_sql = """
SELECT
    B.brand_id,
    B.brand_name,
    COUNT(DISTINCT P.product_id) AS product_count,
    ROUND(AVG(M.sale_price), 0) AS avg_sale_price,
    ROUND(AVG(M.rating), 2) AS avg_rating,
    SUM(M.review_count) AS total_review_count,
    SUM(M.like_count) AS total_like_count
FROM brands B
LEFT JOIN products P
    ON B.brand_id = P.brand_id
LEFT JOIN product_metrics M
    ON P.product_id = M.product_id
GROUP BY B.brand_id, B.brand_name
ORDER BY B.brand_id
"""

mysql_brand_df = pd.read_sql(brand_sql, engine)

pipeline = [
    {
        "$group": {
            "_id": "$brand_id",
            "blog_post_count": {"$sum": 1},
            "unique_blogger_count": {"$addToSet": "$bloggername"},
            "latest_post_date": {"$max": "$postdate"}
        }
    }
]

mongo_result = list(blog_collection.aggregate(pipeline))
mongo_df = pd.DataFrame(mongo_result)

if not mongo_df.empty :
    mongo_df = mongo_df.rename(columns={"_id": "brand_id"})
    mongo_df["unique_blogger_count"] = mongo_df["unique_blogger_count"].apply(len)
else :
    mongo_df = pd.DataFrame(columns=[
        "brand_id", "blog_post_count", "unique_blogger_count", "latest_post_date"
    ])

brand_summary_df = pd.merge(
    mysql_brand_df,
    mongo_df,
    on="brand_id",
    how="left"
)

numeric_cols = [
    "product_count",
    "avg_sale_price",
    "avg_rating",
    "total_review_count",
    "total_like_count",
    "blog_post_count",
    "unique_blogger_count"
]

for col in numeric_cols :
    if col in brand_summary_df.columns :
        brand_summary_df[col] = brand_summary_df[col].fillna(0)

if "latest_post_date" in brand_summary_df.columns :
    brand_summary_df["latest_post_date"] = brand_summary_df["latest_post_date"].fillna("")

brand_summary_df = brand_summary_df.fillna("")

int_cols = [
    "avg_sale_price",
    "total_review_count",
    "total_like_count"
]

for col in int_cols :
    if col in brand_summary_df.columns :
        brand_summary_df[col] = brand_summary_df[col].astype(int)

# blog_posts_detail - 2
docs = list(
    blog_collection.find(
    {},
    {
        "_id": 0,
        "brand_id": 1,
        "brand_name": 1,
        "query": 1,
        "source": 1,
        "title": 1,
        "link": 1,
        "description": 1,
        "bloggername": 1,
        "bloggerlink": 1,
        "postdate": 1,
        "collected_at": 1
    }
  )
)

blog_detail_df = pd.DataFrame(docs)

if not blog_detail_df.empty :
    if "postdate" in blog_detail_df.columns :
        blog_detail_df["postdate"] = pd.to_datetime(blog_detail_df["postdate"], format="%Y%m%d", errors="coerce").dt.strftime("%Y-%m-%d")
        
    if "collected_at" in blog_detail_df.columns :
        blog_detail_df["collected_at"] = pd.to_datetime(blog_detail_df["collected_at"], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")

    blog_detail_df = blog_detail_df.fillna("")
    blog_detail_df = blog_detail_df.sort_values(
        ["brand_id", "postdate"],
         ascending=[True, False]
    )
else :
    blog_detail_df = pd.DataFrame(columns=[
       "brand_id", "brand_name", "query", "source", "title", "link",
        "description", "bloggername", "bloggerlink", "postdate", "collected_at"
    ])

# product_summary - 3
product_sql = """
SELECT
    B.brand_id,
    B.brand_name,
    P.product_id,
    P.product_no,
    P.product_url,
    MAX(M.sale_price) AS sale_price,
    MAX(M.original_price) AS original_price,
    MAX(M.discount_rate) AS discount_rate,
    MAX(M.rating) AS rating,
    MAX(M.review_count) AS review_count,
    MAX(M.like_count) AS like_count,
    MAX(M.crawl_at) AS latest_crawl_at
FROM brands B
LEFT JOIN products P
    ON B.brand_id = P.brand_id
LEFT JOIN product_metrics M
    ON P.product_id = M.product_id
GROUP BY
    B.brand_id,
    B.brand_name,
    P.product_id,
    P.product_no,
    P.product_name,
    P.product_url
ORDER BY B.brand_name, P.product_name
"""

product_summary_df = pd.read_sql(product_sql, engine)

if "latest_crawl_at" in product_summary_df.columns :
    product_summary_df["latest_crawl_at"] = pd.to_datetime(product_summary_df["latest_crawl_at"], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
    product_summary_df["latest_crawl_at"] = product_summary_df["latest_crawl_at"].fillna("")
    
scope = [
    "https://spreadsheets.google.com/feeds",
    "https://www.googleapis.com/auth/drive"
]

creds = ServiceAccountCredentials.from_json_keyfile_name(
    GOOGLE_JSON_FILE,
    scope
)

client = gspread.authorize(creds)
spreadsheet = client.open_by_url(SPREADSHEET_URL)

def get_or_create_worksheet(spreadsheet, title, rows=1000, cols=20) :
    try :
        ws = spreadsheet.worksheet(title)
    except :
        ws = spreadsheet.add_worksheet(title=title, rows=rows, cols=cols)
    return ws

def upload_dataframe_to_sheet(spreadsheet, sheet_name, df, rows=1000, cols=20) :
    ws = get_or_create_worksheet(spreadsheet, sheet_name, rows=rows, cols=cols)
    ws.clear()

    df = df.fillna("")
    data = [df.columns.tolist()] + df.values.tolist()

    ws.update(values=data, range_name="A1")
    print(f"{sheet_name} 탭 업로드 완료")

upload_dataframe_to_sheet(
    spreadsheet,
    "brand_summary",
    brand_summary_df,
    rows=max(len(brand_summary_df) + 10, 1000),
    cols=max(len(brand_summary_df.columns) + 5, 20)
)

upload_dataframe_to_sheet(
    spreadsheet,
    "blog_posts_detail",
    blog_detail_df,
    rows=max(len(blog_detail_df) + 10, 1000),
    cols=max(len(blog_detail_df.columns) + 5, 20)
)

upload_dataframe_to_sheet(
    spreadsheet,
    "product_summary",
    product_summary_df,
    rows=max(len(product_summary_df) + 10, 1000),
    cols=max(len(product_summary_df.columns) + 5, 20)
)

print("모든 탭 업로드 완료")

brand_summary 탭 업로드 완료
blog_posts_detail 탭 업로드 완료
product_summary 탭 업로드 완료
모든 탭 업로드 완료


In [53]:
product_summary_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197 entries, 0 to 196
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   brand_id         197 non-null    int64         
 1   brand_name       197 non-null    object        
 2   product_id       197 non-null    int64         
 3   product_no       197 non-null    object        
 4   product_url      197 non-null    object        
 5   sale_price       197 non-null    int64         
 6   original_price   197 non-null    int64         
 7   discount_rate    197 non-null    int64         
 8   rating           197 non-null    float64       
 9   review_count     197 non-null    int64         
 10  like_count       197 non-null    int64         
 11  latest_crawl_at  197 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(7), object(3)
memory usage: 18.6+ KB


In [48]:
!pip install gspread

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [47]:
!pip install oauth2client

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [45]:
product_summary_df.shape

(197, 12)

In [46]:
product_summary_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197 entries, 0 to 196
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   brand_id         197 non-null    int64         
 1   brand_name       197 non-null    object        
 2   product_id       197 non-null    int64         
 3   product_no       197 non-null    object        
 4   product_url      197 non-null    object        
 5   sale_price       197 non-null    int64         
 6   original_price   197 non-null    int64         
 7   discount_rate    197 non-null    int64         
 8   rating           197 non-null    float64       
 9   review_count     197 non-null    int64         
 10  like_count       197 non-null    int64         
 11  latest_crawl_at  197 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(7), object(3)
memory usage: 18.6+ KB


In [24]:
blog_detail_df.shape

(527, 11)

In [35]:
blog_detail_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 527 entries, 0 to 526
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   brand_id      527 non-null    int64         
 1   brand_name    527 non-null    object        
 2   query         527 non-null    object        
 3   source        527 non-null    object        
 4   title         527 non-null    object        
 5   link          527 non-null    object        
 6   description   527 non-null    object        
 7   bloggername   527 non-null    object        
 8   bloggerlink   527 non-null    object        
 9   postdate      527 non-null    object        
 10  collected_at  527 non-null    datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(9)
memory usage: 45.4+ KB


In [38]:
blog_detail_df

,brand_id,brand_name,query,source,title,link,description,bloggername,bloggerlink,postdate,collected_at
4,1,아디다스,아디다스,naver_blog,김해아울렛 나이키 아디다스 라코스테 세일 추가 할인 20~30프로,https://blog.naver.com/bbiccu6038/224256544357,"나이키, 아디다스, 라코스테 이렇게 세 군데만 집중해서 둘러봤어요. 먼저 아디다스 ...",맛집찾아삼만리,blog.naver.com/bbiccu6038,2026-04-18,2026-04-27 09:56:02.722
1,1,아디다스,아디다스,naver_blog,중국 상해 여행 난징동루 쇼핑 보행자 거리 중티 아디다스 매장,https://blog.naver.com/tnwlsdl702/224249216536,쇼핑거리 아디다스 매장 원래는 난징동루 보행자 거리에서도 플래그쉽 매장으로 가려고 ...,책상에서 즐기는 여행 이야기,blog.naver.com/tnwlsdl702,2026-04-12,2026-04-27 09:56:02.722
3,1,아디다스,아디다스,naver_blog,아디다스 도쿄 JI0183 사이즈 팁과 접지력 좋은 스니커즈...,https://manimo.tistory.com/348,🔗 함께 보면 좋은 상품 할인율 20% [아디다스 공식] 도쿄 JI0183 판매가 ...,manimo,https://manimo.tistory.com/,2026-04-08,2026-04-27 09:56:02.722
0,1,아디다스,아디다스,naver_blog,"아디다스 아디제로 아디오스 프로4 사이즈 팁, 프로3, evo sl...",https://blog.naver.com/nudlelog/224241858954,"저는 주로 아디다스 러닝화를 많이 갖고있는데, 원래 신던 카본화 사이즈가 애매해서 ...",누들로그,blog.naver.com/nudlelog,2026-04-06,2026-04-27 09:56:02.722
2,1,아디다스,아디다스,naver_blog,아디다스 골프화 코드케이오스 필요 대상,https://blog.naver.com/sjrnfl81/224200392483,"&quot;이 포스팅은 네이버 쇼핑 커넥트 활동의 일환으로, 판매 발생 시 수수료를...",또 다른 세상,blog.naver.com/sjrnfl81,2026-03-01,2026-04-27 09:56:02.722
...,...,...,...,...,...,...,...,...,...,...,...
526,200,미나수,미나수,naver_blog,연락 씹고 잠수? 미나수 성훈 못 만남 (솔로지옥5 리유니언 현커),https://blog.naver.com/above_cloud/224183998751,"근데 서로 너무 바빠 가지고 ㅋㅋㅋㅋ (뭐, 마음이 그 정도가 아니었던 거겠지만) ...",과몰입 공간,blog.naver.com/above_cloud,2026-02-14,2026-04-27 09:56:24.456
523,200,미나수,미나수,naver_blog,솔로지옥 리유니언 수빈·희선 손깍지 포착 미나수·민지 화해...,https://blog.naver.com/okjoa012/224180458875,솔로지옥 리유니언 수빈·희선 손깍지 포착 미나수·민지 화해 뒷이야기 현커 공개 넷플...,세계의 영화지도,blog.naver.com/okjoa012,2026-02-11,2026-04-27 09:56:24.456
525,200,미나수,미나수,naver_blog,"민지 원피스 어부바, 주영 냥줍, 수빈 하관설, 미나수 쌍방 울컥...",https://blog.naver.com/redmee2/224180203665,"민지 원피스 어부바, 주영 냥줍, 수빈 하관설, 미나수 쌍방 울컥, 고은 껌딱지 어...",미라공감(美羅共感),blog.naver.com/redmee2,2026-02-11,2026-04-27 09:56:24.456
522,200,미나수,미나수,naver_blog,"12화 최커 5커플! 수빈 희선, 성훈 미나수 빼박 현커 증거?",https://blog.naver.com/dearwinterrain/22417932...,다음 미나수♡성훈은 뉴욕 목격담과 더불어 미나수의 인스타에 성훈의 실루엣 처럼 보이...,더할 나위 없는,blog.naver.com/dearwinterrain,2026-02-10,2026-04-27 09:56:24.456


In [14]:
brand_summary_df.shape

(106, 10)

In [15]:
brand_summary_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106 entries, 0 to 105
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   brand_id              106 non-null    int64  
 1   brand_name            106 non-null    object 
 2   product_count         106 non-null    int64  
 3   avg_sale_price        106 non-null    int64  
 4   avg_rating            106 non-null    float64
 5   total_review_count    106 non-null    int64  
 6   total_like_count      106 non-null    int64  
 7   blog_post_count       106 non-null    int64  
 8   unique_blogger_count  106 non-null    int64  
 9   latest_post_date      106 non-null    object 
dtypes: float64(1), int64(7), object(2)
memory usage: 8.4+ KB


In [7]:
mysql_brand_df

,brand_id,brand_name,product_count,avg_sale_price,avg_rating,total_review_count,total_like_count
0,1,아디다스,12,79797.0,4.97,4660.0,47618.0
1,3,우포스,3,45697.0,4.97,1194.0,1638.0
2,4,비에이유 바이 브라이드앤유,10,266543.0,2.47,728.0,33039.0
3,5,쿠쿠,3,229169.0,3.33,13.0,491.0
4,6,모노로우,2,128440.0,4.95,509.0,16952.0
...,...,...,...,...,...,...,...
101,192,꼼파뇨,1,29744.0,5.00,70.0,2013.0
102,194,웰노운,1,87296.0,0.00,0.0,16.0
103,195,허비쉬,1,102520.0,0.00,0.0,13.0
104,196,로스,1,167200.0,0.00,0.0,114.0


In [8]:
mongo_df

,_id,blog_post_count,unique_blogger_count,latest_post_date
0,103,5,"[PUFF BOWL, 모음의 작은 옷장,, 가성비술사, 노미의 New world, ...",20260411
1,9,5,"[헬륨공방 Helilum in Busan, 몽구의 숟가락 한입로그, 기록장, 큐피아...",20260423
2,108,5,"[어디갓츄의 IT스토리, 라하이맘의 놀이터, 희재맘 희재꼬와 추억쌓기, 기록공간, ...",20260407
3,4,5,"[아기 꾸꾸가 사는 집, 츄르의 육아 아카이브, 블로그랑(bloggrang), 아나...",20260319
4,68,5,"[So Smart , So Real! 소소리, 불완전한 완벽주의자., 제인 블로그,...",20260329
...,...,...,...,...
101,6,5,"[째알라의 패션로그, 직장인 빵듀의 자취 라이프, 어떻게든 사고 만다., 카페24 ...",20260409
102,46,5,"[Youngeday 영이데이, Shoppinglog, 아아는연하게, 하경, '◡'✿...",20260408
103,132,5,"[미토리는 참지않긔♥, 리뷰하는 도토리 상점️️, 미미의 하루, 예둥이맘의 소소한 ...",20250904
104,192,5,"[#Refreshing, 조이 블로그, 끄적끄적 일단 다 적고. 기록하고, 남기고,...",20251106
